[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forestdatapartnership/whisp/blob/feat/timber-next/notebooks/timber_pathway_viewer.ipynb)

# Timber pathway viewer

This notebook builds a web page for exploring the Whisp timber decision tree (`risk_timber`). The tree is on the left and a world map on the right. Click a box in the tree to see where it applies on the map, turn on the datasets behind each question, or draw or upload a plot to see which way it goes through the tree.

To make the page:

1. Add your Earth Engine project id in the settings cell below, or leave it empty for a page without map layers.
2. Run all the cells, and sign in with your Google account when asked (the one that has your Earth Engine project). The first time, it also installs Whisp and asks you to restart the session; restart, then run all again.
3. The page is saved as `timber_pathway_viewer.html`. In Colab it downloads to your computer. Open it in Chrome or Edge, or send it to someone.

Notes:

- The map layers stop loading about a day after the page is built. Run the notebook again for a fresh one.
- The combined layers only show from zoom 9 in. Zoomed out they would look bigger than they are and take minutes to draw. Single datasets show at any zoom.
- Analysing a drawn or uploaded plot uses the Whisp API, which for now only accepts pages on whisp.openforis.org, so it is off in a downloaded page (the page says why). An API key added in the settings can be read by anyone with the file.
- Whisp is an exploration tool: its outcomes are signals, not a legal determination.

In [ ]:
# Settings: fill these in, then run all cells
EE_PROJECT = ""        # your Earth Engine project id, e.g. "ee-yourname" (leave empty for a page without map layers)
NATIONAL = None        # national datasets, as ISO2 codes, e.g. ["br", "cm"]; None for all there are, [] for none
WHISP_API_KEY = ""     # optional, and anyone with the page can read it, so usually leave it empty
INPUT_DATASETS = True  # add each input dataset as its own map layer (needs EE_PROJECT)
AGREEMENT = True       # also add layers where 2+ sources must agree on primary forest 2020 and agriculture after 2020
OUTPUT_HTML = "timber_pathway_viewer.html"

## Build the page
No need to change anything below. Open a cell if you want to see the code.

In [ ]:
#@title Install (Colab) and build the page
import os
import sys

# On a local checkout, use THIS checkout's whisp code (the src folder beside notebooks), so the page is built
# from its own branch even when the active Python is linked to another checkout.
for _src in [os.path.abspath(os.path.join(os.getcwd(), p, "src")) for p in ("..", ".")]:
    if os.path.isdir(os.path.join(_src, "openforis_whisp")):
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

import ee

# Sign in to Earth Engine before loading Whisp. The first time, it asks you to pick a Google account: use the one
# that has your Earth Engine project.
if EE_PROJECT:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

BRANCH = "git+https://github.com/forestdatapartnership/whisp.git@feat/timber-next"
try:
    from openforis_whisp import timber_map
    timber_map.build_timber_viewer  # noqa: B018 (the viewer code is on the feat/timber-next branch)
except ImportError:
    # not installed yet (a fresh Colab session): install it and what it needs, leaving Colab's own packages alone
    !{sys.executable} -m pip install -q "{BRANCH}"
    try:
        from openforis_whisp import timber_map
    except Exception:
        raise SystemExit("Installed. Restart the session (Runtime > Restart session), then run all cells again.")
except AttributeError:
    # an older openforis-whisp is installed: replace just that package
    !{sys.executable} -m pip install -q --force-reinstall --no-deps "{BRANCH}"
    raise SystemExit("Updated openforis-whisp. Restart the session (Runtime > Restart session), then run all cells again.")

html = timber_map.build_timber_viewer(
    ee_project=EE_PROJECT or None,
    national_codes=NATIONAL,
    api_key=WHISP_API_KEY or None,
    input_datasets=INPUT_DATASETS,
    agreement=AGREEMENT,
)
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)
print("Wrote", OUTPUT_HTML)
print("  map layers:", "yes, they stop loading about a day from now" if EE_PROJECT else "none (nothing expires)")
print("  API key in the page:", "yes, anyone with the file can read it" if WHISP_API_KEY else "no, the person opening it can paste their own where the page can reach the API")

In [ ]:
#@title Preview here, and download in Colab
import html as _h
from IPython.display import HTML, display

display(HTML(f'<iframe srcdoc="{_h.escape(html)}" style="width:100%;height:820px;border:1px solid #ccc"></iframe>'))
try:
    from google.colab import files  # noqa
    files.download(OUTPUT_HTML)
except ImportError:
    print("The page is at", OUTPUT_HTML, "(if the preview is blank, open it in a browser)")

## For developers: analyse plots with Whisp on your own computer
Most people can skip this. If you have openforis-whisp installed, you can serve the page from your computer, so drawn or uploaded plots are analysed with that code and your Earth Engine project. The threshold sliders then work too, and so does the pick button (top left of the map): click anywhere to see the pathway and the datasets there. In a terminal, in the folder with the page:

```
python -m openforis_whisp.serve_viewer timber_pathway_viewer.html 8788 your-ee-project
```

Put your Earth Engine project id in place of `your-ee-project`, then open http://127.0.0.1:8788/ in Chrome or Edge. This does not work from Colab.